In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pytorch_lightning as pl
from time import time
from sklearn.model_selection import train_test_split
import pandas as pd

In [2]:
train_df = pd.read_csv("osv-5m/train.csv")

/tmp/ipykernel_3960056/3030339380.py:1: DtypeWarning: Columns (11,27) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("osv-5m/train.csv")


In [3]:
train_df.shape

(4894684, 32)

In [4]:
train_df.head()

,id,latitude,longitude,thumb_original_url,country,sequence,captured_at,lon_bin,lat_bin,cell,...,quadtree_10_50000,quadtree_10_12500,quadtree_10_500,quadtree_10_2500,unique_region,unique_sub-region,unique_city,unique_country,creator_username,creator_id
0,3859149887465501,-43.804769,-176.614093,https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...,NZ,1qOCfogkQ5uE4ZvEf_4n2A,1547559856000,0,8,"(0, 8)",...,0,0,0,0,Chatham Islands_NZ,NaN,Waitangi_NaN_Chatham Islands_NZ,NZ,roadroid,1.113362e+14
1,574181207305439,-43.796611,-176.660483,https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...,NZ,5rdtj6tui6qdlg8q812ul2,1567257056000,0,8,"(0, 8)",...,0,0,0,0,Chatham Islands_NZ,NaN,Waitangi_NaN_Chatham Islands_NZ,NZ,roadroid,1.113362e+14
2,333574322129026,-43.818092,-176.578383,https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...,NZ,nJcst1M2bFUxSOA54CZiy9,1531231715132,0,8,"(0, 8)",...,0,0,0,0,Chatham Islands_NZ,NaN,Waitangi_NaN_Chatham Islands_NZ,NZ,roadroid,1.113362e+14
3,636305258168031,-44.052910,-176.633065,https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...,NZ,BGSwcf0pLCE5bMUWdKTeqk,1662645171414,0,8,"(0, 8)",...,0,0,0,0,Chatham Islands_NZ,NaN,Waitangi_NaN_Chatham Islands_NZ,NZ,roadroid,1.113362e+14
4,166741299029322,-43.748077,-176.329626,https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...,NZ,05pt8HKnLCf47UTGPrxuzB,1531143464444,0,8,"(0, 8)",...,0,0,0,0,Chatham Islands_NZ,NaN,Waitangi_NaN_Chatham Islands_NZ,NZ,roadroid,1.113362e+14


In [5]:
num_examples = len(train_df)
num_examples

4894684

In [6]:
train_df.drop(['unique_region','unique_sub-region','road_index','sequence', 'captured_at', 'lon_bin', 'lat_bin', 'cell','thumb_original_url','creator_id', 'creator_username', 'quadtree_10_5000', 'quadtree_10_25000', 'quadtree_10_1000', 'quadtree_10_50000'], axis=1, inplace=True)

In [7]:
import numpy as np

def MSE(y, yhat):
    return np.mean((y-yhat) ** 2)

def RMSE(y, yhat):
    return np.sqrt(MSE(y, yhat))

def angular_difference(angle1, angle2):
    """
    Compute the shortest angular difference between two angles in degrees.
    Handles wraparound at -180/+180 degrees (e.g., 179 and -179 are 2 degrees apart).
    """
    diff = angle1 - angle2
    # Normalize to [-180, 180]
    diff = (diff + 180) % 360 - 180
    return diff

def circular_mean(angles):
    """
    Compute the circular mean of angles in degrees.
    This properly handles the wraparound at -180/+180 degrees.
    """
    # Convert to radians
    angles_rad = np.deg2rad(angles)
    # Compute mean of sin and cos
    sin_mean = np.mean(np.sin(angles_rad))
    cos_mean = np.mean(np.cos(angles_rad))
    # Convert back to degrees
    mean_rad = np.arctan2(sin_mean, cos_mean)
    return np.rad2deg(mean_rad)

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees).
    Returns distance in kilometers.
    """
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Radius of earth in kilometers
    r = 6371
    
    return c * r

In [8]:
# Bias Regressor
y = train_df[['latitude','longitude']]

# Regular mean for latitude (no wraparound issue)
mean_lat = np.mean(y['latitude'])

# Circular mean for longitude (handles -180/+180 wraparound)
mean_lon = circular_mean(y['longitude'])

print("Bias Regressor Predictions:")
print(f"Mean Latitude: {mean_lat:.6f}°")
print(f"Mean Longitude (circular): {mean_lon:.6f}°")
print()


# NEW METHOD: Haversine distance in kilometers
# This properly accounts for the spherical nature of Earth
distances = haversine_distance(y['latitude'], y['longitude'], mean_lat, mean_lon)
mean_distance = np.mean(distances)
rmse_distance = np.sqrt(np.mean(distances ** 2))

print("Haversine Distance:")
print(f"  Mean absolute error: {mean_distance:.2f} km")
print(f"  RMSE: {rmse_distance:.2f} km")
print()

# Angular errors with wraparound handling
lat_errors = y['latitude'] - mean_lat
lon_errors = angular_difference(y['longitude'], mean_lon)

print("Angular Errors (with longitude wraparound handling):")
print(f"  Latitude RMSE: {np.sqrt(np.mean(lat_errors**2)):.4f} degrees")
print(f"  Longitude RMSE: {np.sqrt(np.mean(lon_errors**2)):.4f} degrees")
print(f"  Combined angular RMSE: {np.sqrt((np.mean(lat_errors**2) + np.mean(lon_errors**2))/2):.4f} degrees")

Bias Regressor Predictions:
Mean Latitude: 32.938720°
Mean Longitude (circular): -11.295957°

Haversine Distance:
  Mean absolute error: 6195.97 km
  RMSE: 7339.67 km

Angular Errors (with longitude wraparound handling):
  Latitude RMSE: 25.3935 degrees
  Longitude RMSE: 75.1507 degrees
  Combined angular RMSE: 56.0912 degrees
